# Libraries

In [18]:
from utilities import *
from modules import *
from framework_stg import *
from evaluation import *
import spacy_component

In [23]:
client = chromadb.PersistentClient(path="./LC_QuAD2.0")
collection = client.get_collection(name="qa_collection")
results = search_chroma(client, "What is the highest temperature in Canada?", top_k=10)
context_text = context(results)
print(context_text)
print(results[0].get('sparql_query'))


What has  location as Ontario? to SPARQL is SELECT DISTINCT ?uri WHERE {?uri <http://dbpedia.org/ontology/location> <http://dbpedia.org/resource/Ontario>  . } 
In how many places are the companies founded in Canada operating? to SPARQL is SELECT DISTINCT COUNT(?uri) WHERE { ?x <http://dbpedia.org/ontology/foundationPlace> <http://dbpedia.org/resource/Canada> . ?x <http://dbpedia.org/property/locations> ?uri  . } 
What river originates in Kingston Ontario? to SPARQL is SELECT DISTINCT ?uri WHERE {?uri <http://dbpedia.org/property/sourceLocation> <http://dbpedia.org/resource/Kingston,_Ontario>  . ?uri <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://dbpedia.org/ontology/River>} 
How many things have made different people living in Canada famous? to SPARQL is SELECT DISTINCT COUNT(?uri) WHERE { ?x <http://dbpedia.org/ontology/nationality> <http://dbpedia.org/resource/Canadians> . ?x <http://dbpedia.org/ontology/knownFor> ?uri  . } 
What is the river whose source is Lake Ontario? 

In [26]:
llm = ChatOpenAI(
    model_name="hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q8_0.gguf",
    base_url="http://localhost:11434/v1",
    openai_api_key="fhguh",
)

In [27]:
llm2 = ChatOpenAI(
    model_name="hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q8_0.gguf",
    base_url="http://localhost:11434/v1",
    openai_api_key="fhguh",
)

In [31]:
df_test = pd.read_csv("queries.csv")

In [32]:
if __name__ == "__main__":
    initial_state = {
        "QUESTION": df_test["question"][0],
        "KB": "dbpedia",
        "URIS": {"entity": df_test["entity"][0], "relation": df_test["relations"][0]},
        "LLM": llm,  # set your LLM object here
        "LLM2" :llm2
    }



#Look Patten

In [33]:
final_state = CHAIN.invoke(initial_state)

I am NER
I am RAG
I am Generation
I am Judge
7.196491003036499
I am SPARQL


In [36]:
final_state["PATTERNS_LIST"]

['PREFIX dbo: <http://dbpedia.org/ontology/> PREFIX res: <http://dbpedia.org/resource/> ASK WHERE { {{Replace with URI}} {{Replace with URI}} {{Replace with URI}} }',
 'PREFIX dbo: <http://dbpedia.org/ontology/> PREFIX res: <http://dbpedia.org/resource/> PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> SELECT DISTINCT ?uri WHERE { ?uri rdf:type {{Replace with URI}} ; {{Replace with URI}} {{Replace with URI}} }',
 'PREFIX dbo: <http://dbpedia.org/ontology/> SELECT ?books WHERE { ?books {{Replace with URI}} <http://dbpedia.org/resource/John_Green_(author)> }',
 'SELECT DISTINCT ?uri WHERE { ?uri a {{Replace with URI}} ; {{Replace with URI}} {{Replace with URI}} }',
 'SELECT DISTINCT ?uri WHERE { {{Replace with URI}} {{Replace with URI}} ?uri }',
 'PREFIX dbo: <http://dbpedia.org/ontology/> PREFIX res: <http://dbpedia.org/resource/> SELECT DISTINCT ?uri WHERE { ?uri {{Replace with URI}} {{Replace with URI}} }',
 'PREFIX dbo: <http://dbpedia.org/ontology/> PREFIX res: <http://dbpe

In [37]:
final_state["FINAL_SPARQL"]

'SELECT DISTINCT ?uri WHERE { res:Robin_Hood wdt:P674 ?x . ?uri wdt:P31 ?x }'

In [38]:
final_state["FINAL_PROMPT"]

'SELECT DISTINCT ?uri WHERE { {{Replace with URI}} {{Replace with URI}} ?uri }'

# **TRAINING**

In [22]:
!ollama list

]11;?\

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


NAME                                                                                          ID              SIZE      MODIFIED     
hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q6_K.gguf                      eb9df1b6b040    12 GB     4 weeks ago     
hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q2_K.gguf                      d366af79c66f    5.8 GB    4 weeks ago     
hf.co/bartowski/Qwen2.5-14B-Instruct-GGUF:Qwen2.5-14B-Instruct-Q4_K_M.gguf                    4746d59a0e20    9.0 GB    4 weeks ago     
hf.co/Qwen/Qwen2.5-14B-Instruct-GGUF:qwen2.5-14b-instruct-q4_k_m-00001-of-00003.gguf          a2909dc4cb13    4.0 GB    4 weeks ago     
hf.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF:qwen2.5-1.5b-instruct-q8_0.gguf                         a3bcfc8f30f6    1.9 GB    7 weeks ago     
hf.co/bartowski/Qwen2.5-7B-Instruct-GGUF:Qwen2.5-7B-Instruct-Q8_0.gguf                        5e4fc2e06fcf    8.1 GB    7 weeks ago     
hf.co/bartowski/microsoft_Phi-4-mini-instruc

In [45]:
llm = ChatOpenAI(
    model_name = "hf.co/Qwen/Qwen2.5-3B-Instruct-GGUF:qwen2.5-3b-instruct-q8_0.gguf",
    base_url="http://loca3lhost:11434/v1",
    openai_api_key="fhguh",
)

In [46]:
import setproctitle
setproctitle.setproctitle("PickA")

In [ ]:
import time
import os
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sacrebleu.metrics import BLEU

# ---- BLEU-1 functions ----
def bleu1_nltk(reference: str, hypothesis: str) -> float:
    """BLEU-1 using NLTK (unigram only)."""
    ref_tokens = [reference.split()]
    hyp_tokens = hypothesis.split()
    smoothie = SmoothingFunction().method1
    return sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)


def bleu1_sacre(reference: str, hypothesis: str) -> float:
    """BLEU-1 using SacreBLEU (unigram only)."""
    bleu = BLEU(max_ngram_order=1)
    result = bleu.corpus_score([hypothesis], [[reference]])
    return result.score / 100.0  # normalize to [0, 1]


# ---- CSV setup ----
csv_file = "QALD_9_ERResultBelu1.csv"

# ---- init ----
y, x = test_pipeline("QALD_9_ER.csv")

if not os.path.exists(csv_file):
    pd.DataFrame(columns=[
        "question", "ground_truth", "final_sparql", "complete_prompts",
        "best_bleu_candidate", "best_f1_candidate",
        "bleu", "bleu_sacre", "bleu_nltk", "bleu4_nltk", "bleu4_sacre",
        "bleu1_nltk", "bleu1_sacre",                          # ← NEW
        "f1", "f1_custom", "f2", "exact_match", "accuracy",
        "sparql_valid", "model"
    ]).to_csv(csv_file, index=False)

# ---- OPTIONAL: AUTO-RESUME ----
if os.path.exists(csv_file):
    existing = pd.read_csv(csv_file)
    start_idx = len(existing)
else:
    start_idx = 0

# ---- LIMIT RANGE ----
max_samples = 1000
end_idx = min(len(x), start_idx + max_samples)

end_idx = len(x)  # or set to 1000 if you want a limit

# ---- metrics init ----
t_bleu = t_belu2 = t_belu4 = 0
t_em = t_f1 = t_f2 = 0
t_bleu_sacre = t_bleu_nltk = 0
t_bleu4_nltk = t_bleu4_sacre = 0
t_bleu1_nltk = t_bleu1_sacre = 0                              # ← NEW
t_f1_lib = 0
t_accuracy = 0
validate_sparql = 0

t_f1_best = t_belu_best = 0
t_bleu_best_f1_cand = 0
t_f1_best_bleu_cand = 0

start = time.time()

# ---- LOOP ----
for items in range(start_idx, end_idx):

    count = items - start_idx + 1  # correct averaging

    initial_state = {
        "QUESTION": x[items],
        "KB": "wikidata",
        "URIS": {
            "entity": df_test["entities"][items],
            "relation": df_test["relations"][items]
        },
        "LLM": llm,
        "LLM2": llm2
    }
    final_state = CHAIN.invoke(initial_state)

    final_candidates = final_state["COMPLETED_PROMPTS"]
    output = final_state["FINAL_SPARQL"]

    print(f"[GT] : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")

    # ---- timing ----
    end = time.time()
    infer_time = end - start

    # ---- metrics ----
    scores = evaluate_texts(y[items], output)
    results = evaluate_pair(y[items], output)

    f2 = f1_score(output, y[items])
    score = 1  # placeholder

    t_belu2       += bleu_score(y[items], output)
    t_bleu        += scores["BLEU"]
    t_em          += scores["Exact Match"]
    t_f1          += scores["F1"]
    t_f2          += f2
    t_bleu_sacre  += results["BLEU_sacre"]
    t_bleu_nltk   += results["BLEU_nltk"]
    t_f1_lib      += results["F1"]

    t_bleu4_nltk  += bleu4_nltk(y[items], output)
    t_bleu4_sacre += bleu4_sacre(y[items], output)

    # ---- BLEU-1 ---- (NEW)
    b1_nltk  = bleu1_nltk(y[items], output)
    b1_sacre = bleu1_sacre(y[items], output)
    t_bleu1_nltk  += b1_nltk
    t_bleu1_sacre += b1_sacre

    best_results = best_candidate_by_bleu_and_f1(final_candidates, y[items])

    t_f1_best          += best_results["f1"]
    t_belu_best        += best_results["bleu"]
    t_bleu_best_f1_cand += best_results["bleu_of_best_f1"]
    t_f1_best_bleu_cand += best_results["f1_of_best_bleu"]

    t_accuracy += score

    # ---- SPARQL validation ----
    true, cmt = check_sparql_query_validity(output, "wikidata")
    if true:
        validate_sparql += 1

    # ---- logging ----
    print(
        f"epoch: {items+1}, "
        f"bleu2: {t_belu2/count:.4f}, "
        f"bleu: {t_bleu/count:.4f}, "
        f"em: {t_em}, "
        f"f1: {t_f1/count:.4f}, "
        f"f2: {t_f2/count:.4f}, "
        f"accuracy: {t_accuracy/count:.4f}, "
        f"time: {infer_time/count:.4f}s, "
        f"valid_sparql: {validate_sparql}"
    )

    print(
        f"BLEU_sacre: {t_bleu_sacre/count:.4f}, "
        f"BLEU_nltk: {t_bleu_nltk/count:.4f}, "
        f"BLEU4_nltk: {t_bleu4_nltk/count:.4f}, "
        f"BLEU4_sacre: {t_bleu4_sacre/count:.4f}, "
        f"F1_custom: {t_f1_lib/count:.4f}, "
        f"Model: {llm.model_name}"
    )

    print(
        f"BLEU1_nltk: {t_bleu1_nltk/count:.4f}, "           # ← NEW
        f"BLEU1_sacre: {t_bleu1_sacre/count:.4f}"           # ← NEW
    )

    print(
        f"best_bleu: {t_belu_best/count:.4f}, "
        f"best_f1: {t_f1_best_bleu_cand/count:.4f}"
    )

    print("=" * 100)

    # ---- row to save ----
    row = {
        "question":         x[items],
        "ground_truth":     y[items],
        "final_sparql":     output,
        "completed_prompts": " || ".join(final_candidates),

        "best_bleu_candidate": best_results.get("best_bleu_candidate"),
        "best_f1_candidate":   best_results.get("best_f1_candidate"),

        "bleu":       scores["BLEU"],
        "bleu_sacre": results["BLEU_sacre"],
        "bleu_nltk":  results["BLEU_nltk"],
        "bleu4_nltk": bleu4_nltk(y[items], output),
        "bleu4_sacre": bleu4_sacre(y[items], output),

        "bleu1_nltk":  b1_nltk,                              # ← NEW
        "bleu1_sacre": b1_sacre,                             # ← NEW

        "f1":       scores["F1"],
        "f1_custom": results["F1"],
        "f2":        f2,

        "exact_match": scores["Exact Match"],
        "accuracy":    score,
        "final_context": final_state["CONTEXT"],

        "sparql_valid": true,
        "model":        llm.model_name
    }

    # ---- write to CSV ----
    pd.DataFrame([row]).to_csv(
        csv_file,
        mode='a',
        header=False,
        index=False
    )

    print("Saved to CSV")
    print("=" * 100)

In [ ]:
import time
import os
import pandas as pd
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from sacrebleu.metrics import BLEU


# ========================================================
# METRIC FUNCTIONS
# ========================================================

def bleu1_nltk(reference: str, hypothesis: str) -> float:
    """BLEU-1 using NLTK (unigram only)."""
    ref_tokens = [reference.split()]
    hyp_tokens = hypothesis.split()
    smoothie = SmoothingFunction().method1
    return sentence_bleu(ref_tokens, hyp_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothie)


def bleu1_sacre(reference: str, hypothesis: str) -> float:
    """BLEU-1 using SacreBLEU (unigram only)."""
    bleu = BLEU(max_ngram_order=1)
    result = bleu.corpus_score([hypothesis], [[reference]])
    return result.score / 100.0


def token_level_f1(reference: str, hypothesis: str) -> dict:
    """
    Token-level F1 (SQuAD-style) using token frequencies.

    Precision = common_tokens / total tokens in hypothesis
    Recall    = common_tokens / total tokens in reference
    F1        = (2 * Precision * Recall) / (Precision + Recall)

    common_tokens counts overlapping tokens with their frequencies:
    e.g. reference="a a b", hypothesis="a b b"
         common = min(2,1) + min(1,2) = 1 + 1 = 2
    """
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)

    # overlapping tokens respecting frequency
    common_tokens = sum((ref_counts & hyp_counts).values())

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


# ========================================================
# CSV SETUP
# ========================================================

csv_file = "QALD_9_ERR_Token.csv"

# ---- init ----
y, x = test_pipeline("QALD_9_ER.csv")

if not os.path.exists(csv_file):
    pd.DataFrame(columns=[
        "question", "ground_truth", "final_sparql", "completed_prompts",
        "best_bleu_candidate", "best_f1_candidate",
        "bleu", "bleu_sacre", "bleu_nltk", "bleu4_nltk", "bleu4_sacre",
        "bleu1_nltk", "bleu1_sacre",
        "token_precision", "token_recall", "token_f1",
        "f1", "f1_custom", "f2", "exact_match", "accuracy",
        "final_context", "sparql_valid", "model"
    ]).to_csv(csv_file, index=False)

# ========================================================
# AUTO-RESUME
# ========================================================

if os.path.exists(csv_file):
    existing  = pd.read_csv(csv_file)
    start_idx = len(existing)
else:
    start_idx = 0

# ========================================================
# LIMIT RANGE
# ========================================================

max_samples = 1000
end_idx     = min(len(x), start_idx + max_samples)
end_idx     = len(x)   # comment out if you want the 1000-sample limit

# ========================================================
# METRICS ACCUMULATORS
# ========================================================

t_bleu        = 0
t_belu2       = 0
t_belu4       = 0
t_em          = 0
t_f1          = 0
t_f2          = 0
t_bleu_sacre  = 0
t_bleu_nltk   = 0
t_bleu4_nltk  = 0
t_bleu4_sacre = 0
t_bleu1_nltk  = 0
t_bleu1_sacre = 0
t_f1_lib      = 0
t_accuracy    = 0
validate_sparql = 0

t_token_precision = 0
t_token_recall    = 0
t_token_f1        = 0

t_f1_best           = 0
t_belu_best         = 0
t_bleu_best_f1_cand = 0
t_f1_best_bleu_cand = 0

start = time.time()

# ========================================================
# MAIN LOOP
# ========================================================

for items in range(start_idx, end_idx):

    count = items - start_idx + 1  # correct rolling average denominator

    # ---- build state and invoke chain ----
    initial_state = {
        "QUESTION": x[items],
        "KB": "wikidata",
        "URIS": {
            "entity":   df_test["entities"][items],
            "relation": df_test["relations"][items]
        },
        "LLM":  llm,
        "LLM2": llm2
    }
    final_state = CHAIN.invoke(initial_state)

    final_candidates = final_state["COMPLETED_PROMPTS"]
    output           = final_state["FINAL_SPARQL"]

    print(f"[GT]           : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")

    # ---- timing ----
    end        = time.time()
    infer_time = end - start

    # ---- standard metrics ----
    scores  = evaluate_texts(y[items], output)
    results = evaluate_pair(y[items], output)
    f2      = f1_score(output, y[items])
    score   = 1  # placeholder accuracy

    t_belu2       += bleu_score(y[items], output)
    t_bleu        += scores["BLEU"]
    t_em          += scores["Exact Match"]
    t_f1          += scores["F1"]
    t_f2          += f2
    t_bleu_sacre  += results["BLEU_sacre"]
    t_bleu_nltk   += results["BLEU_nltk"]
    t_f1_lib      += results["F1"]
    t_bleu4_nltk  += bleu4_nltk(y[items], output)
    t_bleu4_sacre += bleu4_sacre(y[items], output)

    # ---- BLEU-1 ----
    b1_nltk  = bleu1_nltk(y[items], output)
    b1_sacre = bleu1_sacre(y[items], output)
    t_bleu1_nltk  += b1_nltk
    t_bleu1_sacre += b1_sacre

    # ---- Token-level Precision / Recall / F1 ----
    tok = token_level_f1(y[items], output)
    t_token_precision += tok["precision"]
    t_token_recall    += tok["recall"]
    t_token_f1        += tok["f1"]

    # ---- best candidate metrics ----
    best_results         = best_candidate_by_bleu_and_f1(final_candidates, y[items])
    t_f1_best           += best_results["f1"]
    t_belu_best         += best_results["bleu"]
    t_bleu_best_f1_cand += best_results["bleu_of_best_f1"]
    t_f1_best_bleu_cand += best_results["f1_of_best_bleu"]

    t_accuracy += score

    # ---- SPARQL validation ----
    true, cmt = check_sparql_query_validity(output, "wikidata")
    if true:
        validate_sparql += 1

    # ========================================================
    # LOGGING
    # ========================================================

    print(
        f"epoch: {items+1} | "
        f"bleu2: {t_belu2/count:.4f} | "
        f"bleu: {t_bleu/count:.4f} | "
        f"em: {t_em} | "
        f"f1: {t_f1/count:.4f} | "
        f"f2: {t_f2/count:.4f} | "
        f"accuracy: {t_accuracy/count:.4f} | "
        f"time: {infer_time/count:.4f}s | "
        f"valid_sparql: {validate_sparql}"
    )

    print(
        f"BLEU_sacre: {t_bleu_sacre/count:.4f} | "
        f"BLEU_nltk: {t_bleu_nltk/count:.4f} | "
        f"BLEU4_nltk: {t_bleu4_nltk/count:.4f} | "
        f"BLEU4_sacre: {t_bleu4_sacre/count:.4f} | "
        f"F1_custom: {t_f1_lib/count:.4f} | "
        f"Model: {llm.model_name}"
    )

    print(
        f"BLEU1_nltk: {t_bleu1_nltk/count:.4f} | "
        f"BLEU1_sacre: {t_bleu1_sacre/count:.4f}"
    )

    print(
        f"Token_Precision: {t_token_precision/count:.4f} | "
        f"Token_Recall: {t_token_recall/count:.4f} | "
        f"Token_F1: {t_token_f1/count:.4f}"
    )

    print(
        f"best_bleu: {t_belu_best/count:.4f} | "
        f"best_f1: {t_f1_best_bleu_cand/count:.4f}"
    )

    print("=" * 100)

    # ========================================================
    # SAVE ROW TO CSV
    # ========================================================

    row = {
        "question":          x[items],
        "ground_truth":      y[items],
        "final_sparql":      output,
        "completed_prompts": " || ".join(final_candidates),

        "best_bleu_candidate": best_results.get("best_bleu_candidate"),
        "best_f1_candidate":   best_results.get("best_f1_candidate"),

        "bleu":        scores["BLEU"],
        "bleu_sacre":  results["BLEU_sacre"],
        "bleu_nltk":   results["BLEU_nltk"],
        "bleu4_nltk":  bleu4_nltk(y[items], output),
        "bleu4_sacre": bleu4_sacre(y[items], output),

        "bleu1_nltk":  b1_nltk,
        "bleu1_sacre": b1_sacre,

        "token_precision": tok["precision"],
        "token_recall":    tok["recall"],
        "token_f1":        tok["f1"],

        "f1":        scores["F1"],
        "f1_custom": results["F1"],
        "f2":        f2,

        "exact_match":   scores["Exact Match"],
        "accuracy":      score,
        "final_context": final_state["CONTEXT"],

        "sparql_valid": true,
        "model":        llm.model_name
    }

    pd.DataFrame([row]).to_csv(
        csv_file,
        mode='a',
        header=False,
        index=False
    )

    print("Saved to CSV")
    print("=" * 100)

In [ ]:
import time
import os
import pandas as pd
from collections import Counter
from sacrebleu.metrics import BLEU


# ========================================================
# METRIC FUNCTIONS
# ========================================================

def bleu1_sacre(reference: str, hypothesis: str) -> float:
    """BLEU-1 using SacreBLEU (unigram only)."""
    bleu = BLEU(max_ngram_order=1)
    result = bleu.corpus_score([hypothesis], [[reference]])
    return result.score / 100.0


def token_level_f1(reference: str, hypothesis: str) -> dict:
    """Token-level F1 (SQuAD-style) using token frequencies."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)

    common_tokens = sum((ref_counts & hyp_counts).values())

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


# ========================================================
# CSV SETUP
# ========================================================

csv_file = "QALD_9_ERR_Token.csv"

y, x = test_pipeline("QALD_9_ER.csv")

if not os.path.exists(csv_file):
    pd.DataFrame(columns=[
        "question", "ground_truth", "final_sparql",
        "bleu1_sacre",
        "token_precision", "token_recall", "token_f1",
        "sparql_valid", "model"
    ]).to_csv(csv_file, index=False)


# ========================================================
# AUTO-RESUME
# ========================================================

if os.path.exists(csv_file):
    existing  = pd.read_csv(csv_file)
    start_idx = len(existing)
else:
    start_idx = 0

# ========================================================
# LIMIT RANGE
# ========================================================

end_idx = len(x)

# ========================================================
# METRICS ACCUMULATORS
# ========================================================

t_bleu1_sacre     = 0
t_token_precision = 0
t_token_recall    = 0
t_token_f1        = 0
validate_sparql   = 0

start = time.time()

# ========================================================
# MAIN LOOP
# ========================================================

for items in range(start_idx, end_idx):

    count = items - start_idx + 1

    # ---- build state and invoke chain ----
    initial_state = {
        "QUESTION": x[items],
        "KB": "wikidata",
        "URIS": {
            "entity":   df_test["entities"][items],
            "relation": df_test["relations"][items]
        },
        "LLM":  llm,
        "LLM2": llm2
    }
    final_state = CHAIN.invoke(initial_state)
    output      = final_state["FINAL_SPARQL"]

    print(f"[GT]           : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")

    # ---- timing ----
    infer_time = time.time() - start

    # ---- BLEU SacreBLEU ----
    b1_sacre       = bleu1_sacre(y[items], output)
    t_bleu1_sacre += b1_sacre

    # ---- Token-level F1 ----
    tok                = token_level_f1(y[items], output)
    t_token_precision += tok["precision"]
    t_token_recall    += tok["recall"]
    t_token_f1        += tok["f1"]

    # ---- SPARQL validation ----
    true, _ = check_sparql_query_validity(output, "wikidata")
    if true:
        validate_sparql += 1

    # ========================================================
    # LOGGING
    # ========================================================

    print(
        f"epoch: {items+1} | "
        f"BLEU1_sacre: {t_bleu1_sacre/count:.4f} | "
        f"Token_Precision: {t_token_precision/count:.4f} | "
        f"Token_Recall: {t_token_recall/count:.4f} | "
        f"Token_F1: {t_token_f1/count:.4f} | "
        f"valid_sparql: {validate_sparql} | "
        f"time: {infer_time/count:.4f}s"
    )
    print("=" * 100)

    # ========================================================
    # SAVE ROW TO CSV
    # ========================================================

    row = {
        "question":        x[items],
        "ground_truth":    y[items],
        "final_sparql":    output,
        "bleu1_sacre":     b1_sacre,
        "token_precision": tok["precision"],
        "token_recall":    tok["recall"],
        "token_f1":        tok["f1"],
        "sparql_valid":    true,
        "model":           llm.model_name
    }

    pd.DataFrame([row]).to_csv(csv_file, mode='a', header=False, index=False)

    print("Saved to CSV")
    print("=" * 100)

In [ ]:
import time
import os
import pandas as pd
from collections import Counter
from sacrebleu.metrics import BLEU


# ========================================================
# METRIC FUNCTIONS
# ========================================================

def bleu_sacre(reference: str, hypothesis: str) -> float:
    """Sentence-level BLEU using SacreBLEU."""
    bleu = BLEU()
    result = bleu.sentence_score(hypothesis, [reference])
    return result.score / 100.0

def token_level_f1(reference: str, hypothesis: str) -> dict:
    """Token-level F1 (SQuAD-style) using token frequencies."""
    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)

    common_tokens = sum((ref_counts & hyp_counts).values())

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (2 * precision * recall) / (precision + recall)

    return {"precision": precision, "recall": recall, "f1": f1}


# ========================================================
# CSV SETUP
# ========================================================

csv_file = "QALD_9_ERR_Token1.csv"

y, x = test_pipeline("QALD_9_ER.csv")

if not os.path.exists(csv_file):
    pd.DataFrame(columns=[
        "question", "ground_truth", "final_sparql",
        "bleu_sacre",
        "token_precision", "token_recall", "token_f1",
        "sparql_valid", "model"
    ]).to_csv(csv_file, index=False)


# ========================================================
# AUTO-RESUME
# ========================================================

if os.path.exists(csv_file):
    existing  = pd.read_csv(csv_file)
    start_idx = len(existing)
else:
    start_idx = 0

# ========================================================
# LIMIT RANGE
# ========================================================

end_idx = len(x)

# ========================================================
# METRICS ACCUMULATORS
# ========================================================

t_bleu_sacre      = 0
t_token_precision = 0
t_token_recall    = 0
t_token_f1        = 0
validate_sparql   = 0

start = time.time()

# ========================================================
# MAIN LOOP
# ========================================================

for items in range(start_idx, end_idx):

    count = items - start_idx + 1

    # ---- build state and invoke chain ----
    initial_state = {
        "QUESTION": x[items],
        "KB": "wikidata",
        "URIS": {
            "entity":   df_test["entities"][items],
            "relation": df_test["relations"][items]
        },
        "LLM":  llm,
        "LLM2": llm2
    }
    final_state = CHAIN.invoke(initial_state)
    output      = final_state["FINAL_SPARQL"]

    print(f"[GT]           : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")

    # ---- timing ----
    infer_time = time.time() - start

    # ---- BLEU SacreBLEU ----
    b_sacre       = bleu_sacre(y[items], output)
    t_bleu_sacre += b_sacre

    # ---- Token-level F1 ----
    tok                = token_level_f1(y[items], output)
    t_token_precision += tok["precision"]
    t_token_recall    += tok["recall"]
    t_token_f1        += tok["f1"]

    # ---- SPARQL validation ----
    true, _ = check_sparql_query_validity(output, "wikidata")
    if true:
        validate_sparql += 1

    # ========================================================
    # LOGGING
    # ========================================================

    print(
        f"epoch: {items+1} | "
        f"BLEU_sacre: {t_bleu_sacre/count:.4f} | "
        f"Token_Precision: {t_token_precision/count:.4f} | "
        f"Token_Recall: {t_token_recall/count:.4f} | "
        f"Token_F1: {t_token_f1/count:.4f} | "
        f"valid_sparql: {validate_sparql} | "
        f"time: {infer_time/count:.4f}s"
    )
    print("=" * 100)

    # ========================================================
    # SAVE ROW TO CSV
    # ========================================================

    row = {
        "question":        x[items],
        "ground_truth":    y[items],
        "final_sparql":    output,
        "bleu_sacre":      b_sacre,
        "token_precision": tok["precision"],
        "token_recall":    tok["recall"],
        "token_f1":        tok["f1"],
        "sparql_valid":    true,
        "model":           llm.model_name
    }

    pd.DataFrame([row]).to_csv(csv_file, mode='a', header=False, index=False)

    print("Saved to CSV")
    print("=" * 100)

In [ ]:
# ========================================================
# IMPORTS
# ========================================================

import time
import os
import re
import string
import pandas as pd

from collections import Counter

from sacrebleu.metrics import BLEU
from nltk.translate.bleu_score import sentence_bleu


# ========================================================
# SPARQL NORMALIZATION
# ========================================================

def normalize_sparql(text: str) -> str:

    if text is None:
        return ""

    for kw in [
        'SELECT', 'DISTINCT', 'WHERE', 'FILTER',
        'OPTIONAL', 'UNION', 'LIMIT', 'OFFSET',
        'SERVICE', 'VALUES', 'BIND', 'MINUS',
        'HAVING', 'ASK', 'CONSTRUCT', 'DESCRIBE',
    ]:
        text = re.sub(
            rf'\b{kw}\b',
            kw.lower(),
            text,
            flags=re.IGNORECASE
        )

    text = re.sub(
        r'\bORDER\s+BY\b',
        'order by',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(
        r'\bGROUP\s+BY\b',
        'group by',
        text,
        flags=re.IGNORECASE
    )

    text = re.sub(r'\{', ' { ', text)
    text = re.sub(r'\}', ' } ', text)
    text = re.sub(r'\(', ' ( ', text)
    text = re.sub(r'\)', ' ) ', text)

    text = re.sub(
        r'(?<![:/\w])\.(?=\s|\}|$)',
        ' . ',
        text
    )

    text = re.sub(
        r'(\w)\.(\s|\}|$)',
        r'\1 . \2',
        text
    )

    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ========================================================
# SGPT NORMALIZATION
# ========================================================

RE_ART  = re.compile(r'\b(a|an|the)\b')
RE_PUNC = re.compile(r'[!"#$%&()*+,-./:;<=>?@\[\]\\^`{|}~_\']')


def _remove_articles(t):
    return RE_ART.sub(' ', t)


def _white_space_fix(t):
    return ' '.join(t.split())


def _remove_punc(t):
    return RE_PUNC.sub(' ', t)


def _lower(t):
    return t.lower()


def normalize(text: str) -> str:
    """
    Lower text and remove punctuation,
    articles and extra whitespace.
    """
    return _white_space_fix(
        _remove_articles(
            _remove_punc(
                _lower(text)
            )
        )
    )


# ========================================================
# ORIGINAL METRICS
# ========================================================

def bleu_sacre_normal(reference: str, hypothesis: str) -> float:

    bleu = BLEU(effective_order=True)

    result = bleu.sentence_score(
        hypothesis,
        [reference]
    )

    return result.score / 100.0


def bleu_sacre_normalized(reference: str, hypothesis: str) -> float:

    bleu = BLEU(effective_order=True)

    result = bleu.sentence_score(
        normalize(hypothesis),
        [normalize(reference)]
    )

    return result.score / 100.0


def bleu_1(reference: str, hypothesis: str) -> float:

    bleu = BLEU(
        effective_order=True,
        max_ngram_order=1
    )

    result = bleu.sentence_score(
        hypothesis,
        [reference]
    )

    return result.score / 100.0


def token_level_f1(reference: str, hypothesis: str) -> dict:

    ref_tokens = reference.split()
    hyp_tokens = hypothesis.split()

    if not ref_tokens or not hyp_tokens:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }

    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)

    common_tokens = sum(
        (ref_counts & hyp_counts).values()
    )

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ========================================================
# CORRECTED F1
# ========================================================

def corrected_f1(reference: str, hypothesis: str) -> dict:

    ref_norm = normalize_sparql(reference)
    hyp_norm = normalize_sparql(hypothesis)

    ref_tokens = ref_norm.split()
    hyp_tokens = hyp_norm.split()

    if not ref_tokens and not hyp_tokens:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0
        }

    if not ref_tokens or not hyp_tokens:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }

    ref_counts = Counter(ref_tokens)
    hyp_counts = Counter(hyp_tokens)

    common_tokens = sum(
        (ref_counts & hyp_counts).values()
    )

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ========================================================
# CORRECTED BLEU
# ========================================================

def corrected_bleu(reference: str, hypothesis: str) -> float:

    bleu = BLEU(effective_order=True)

    result = bleu.sentence_score(
        normalize_sparql(hypothesis),
        [normalize_sparql(reference)]
    )

    return result.score / 100.0


def corrected_bleu_1(reference: str, hypothesis: str) -> float:

    bleu = BLEU(
        effective_order=True,
        max_ngram_order=1
    )

    result = bleu.sentence_score(
        normalize_sparql(hypothesis),
        [normalize_sparql(reference)]
    )

    return result.score / 100.0


# ========================================================
# SGPT F1
# ========================================================

def sgpt_f1(reference: str, hypothesis: str) -> dict:

    ref_tokens = normalize(reference).split()
    hyp_tokens = normalize(hypothesis).split()

    if not ref_tokens and not hyp_tokens:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0
        }

    if not ref_tokens or not hyp_tokens:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }

    common_tokens = sum(
        (
            Counter(ref_tokens)
            &
            Counter(hyp_tokens)
        ).values()
    )

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ========================================================
# SGPT BLEU
# NO NORMALIZATION
# follows original BLEU Metric class
# ========================================================

def sgpt_bleu(reference: str, hypothesis: str) -> float:

    hyp_tokens = hypothesis.split()
    ref_tokens = reference.split()

    bleu = sentence_bleu(
        [ref_tokens],
        hyp_tokens
    )

    return bleu


# ========================================================
# CSV SETUP
# ========================================================

csv_file = "QALD_9_ERR_STG_FINAL.csv"

y, x = test_pipeline("QALD_9_ER.csv")

if not os.path.exists(csv_file):

    pd.DataFrame(columns=[

        "question",
        "ground_truth",
        "final_sparql",

        "bleu_normal",
        "bleu_normalized",
        "bleu_1",

        "token_precision",
        "token_recall",
        "token_f1",

        "sparql_token_precision",
        "sparql_token_recall",
        "sparql_token_f1",

        "sparql_bleu",
        "sparql_bleu_1",

        "corrected_precision",
        "corrected_recall",
        "corrected_f1",

        "corrected_bleu",
        "corrected_bleu_1",

        "sgpt_precision",
        "sgpt_recall",
        "sgpt_f1",
        "sgpt_bleu",

        "sparql_valid",
        "model"

    ]).to_csv(csv_file, index=False)


# ========================================================
# AUTO RESUME
# ========================================================

if os.path.exists(csv_file):

    existing  = pd.read_csv(csv_file)
    start_idx = len(existing)

else:

    start_idx = 0


end_idx = len(x)


# ========================================================
# METRIC ACCUMULATORS
# ========================================================

t_bleu_normal            = 0
t_bleu_normalized        = 0
t_bleu_1                 = 0

t_token_precision        = 0
t_token_recall           = 0
t_token_f1               = 0

t_sparql_token_precision = 0
t_sparql_token_recall    = 0
t_sparql_token_f1        = 0

t_sparql_bleu            = 0
t_sparql_bleu_1          = 0

validate_sparql          = 0

t_corrected_precision    = 0
t_corrected_recall       = 0
t_corrected_f1           = 0

t_corrected_bleu         = 0
t_corrected_bleu_1       = 0

t_sgpt_precision         = 0
t_sgpt_recall            = 0
t_sgpt_f1                = 0
t_sgpt_bleu              = 0

validate_sparql          = 0


# ========================================================
# TRAIN LOOP
# ========================================================

start = time.time()

for items in range(start_idx, end_idx):

    count = items - start_idx + 1

    initial_state = {

        "QUESTION": x[items],

        "KB": "dbpedia",

        "URIS": {
            "entity":   df_test["entities"][items],
            "relation": df_test["relations"][items]
        },

        "LLM":  llm,
        "LLM2": llm2
    }

    final_state = CHAIN.invoke(initial_state)

    output = final_state["FINAL_SPARQL"]

    print(f"[GT]           : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")
    print(f"model name {llm.model_name}")

    infer_time = time.time() - start


    # ====================================================
    # ORIGINAL METRICS
    # ====================================================

    b_normal = bleu_sacre_normal(
        y[items],
        output
    )
    true, _ = check_sparql_query_validity(output, "dbpedia")
    if true:
        validate_sparql += 1

    t_bleu_normal += b_normal


    b_normalized = bleu_sacre_normalized(
        y[items],
        output
    )

    t_bleu_normalized += b_normalized


    b_1 = bleu_1(
        y[items],
        output
    )

    t_bleu_1 += b_1


    tok = token_level_f1(
        y[items],
        output
    )

    t_token_precision += tok["precision"]
    t_token_recall    += tok["recall"]
    t_token_f1        += tok["f1"]


    tok_sparql = 1

    t_sparql_token_precision += tok_sparql["precision"]
    t_sparql_token_recall    += tok_sparql["recall"]
    t_sparql_token_f1        += tok_sparql["f1"]


    b_sparql = bleu_sparql(
        y[items],
        output
    )

    t_sparql_bleu += b_sparql


    b_sparql_1 = bleu_sparql_1(
        y[items],
        output
    )

    t_sparql_bleu_1 += b_sparql_1


    # ====================================================
    # CORRECTED METRICS
    # ====================================================

    cor_f1 = corrected_f1(
        y[items],
        output
    )

    t_corrected_precision += cor_f1["precision"]
    t_corrected_recall    += cor_f1["recall"]
    t_corrected_f1        += cor_f1["f1"]


    cor_bleu = corrected_bleu(
        y[items],
        output
    )

    t_corrected_bleu += cor_bleu


    cor_bleu_1 = corrected_bleu_1(
        y[items],
        output
    )

    t_corrected_bleu_1 += cor_bleu_1


    # ====================================================
    # SGPT METRICS
    # ====================================================

    s_f1 = sgpt_f1(
        y[items],
        output
    )

    t_sgpt_precision += s_f1["precision"]
    t_sgpt_recall    += s_f1["recall"]
    t_sgpt_f1        += s_f1["f1"]


    s_bleu = sgpt_bleu(
        y[items],
        output
    )

    t_sgpt_bleu += s_bleu


    # ====================================================
    # SPARQL VALIDATION
    # ====================================================

    true, _ = check_sparql_query_validity(
        output,
        "dbpedia"
    )

    if true:
        validate_sparql += 1


    # ====================================================
    # LOGGING
    # ====================================================

    print(
        f"epoch: {items+1} | "
        f"BLEU_normal: {t_bleu_normal/count:.4f} | "
        f"BLEU_normalized: {t_bleu_normalized/count:.4f} | "
        f"BLEU_1: {t_bleu_1/count:.4f} | "
        f"Token_Precision: {t_token_precision/count:.4f} | "
        f"Token_Recall: {t_token_recall/count:.4f} | "
        f"Token_F1: {t_token_f1/count:.4f} | "
        f"SPARQL_Precision: {t_sparql_token_precision/count:.4f} | "
        f"SPARQL_Recall: {t_sparql_token_recall/count:.4f} | "
        f"SPARQL_F1: {t_sparql_token_f1/count:.4f} | "
        f"SPARQL_BLEU: {t_sparql_bleu/count:.4f} | "
        f"SPARQL_BLEU_1: {t_sparql_bleu_1/count:.4f} | "
        f"Corrected_Precision: {t_corrected_precision/count:.4f} | "
        f"Corrected_Recall: {t_corrected_recall/count:.4f} | "
        f"Corrected_F1: {t_corrected_f1/count:.4f} | "
        f"Corrected_BLEU: {t_corrected_bleu/count:.4f} | "
        f"Corrected_BLEU_1: {t_corrected_bleu_1/count:.4f} | "
        f"SGPT_Precision: {t_sgpt_precision/count:.4f} | "
        f"SGPT_Recall: {t_sgpt_recall/count:.4f} | "
        f"SGPT_F1: {t_sgpt_f1/count:.4f} | "
        f"SGPT_BLEU: {t_sgpt_bleu/count:.4f} | "
        f"valid_sparql: {validate_sparql} | "
        f"time: {infer_time/count:.4f}s"
    )

    print("=" * 100)


    # ====================================================
    # SAVE ROW
    # ====================================================

    row = {

        "question":               x[items],
        "ground_truth":           y[items],
        "final_sparql":           output,

        "bleu_normal":            b_normal,
        "bleu_normalized":        b_normalized,
        "bleu_1":                 b_1,

        "token_precision":        tok["precision"],
        "token_recall":           tok["recall"],
        "token_f1":               tok["f1"],

        "sparql_token_precision": tok_sparql["precision"],
        "sparql_token_recall":    tok_sparql["recall"],
        "sparql_token_f1":        tok_sparql["f1"],

        "sparql_bleu":            b_sparql,
        "sparql_bleu_1":          b_sparql_1,

        "corrected_precision":    cor_f1["precision"],
        "corrected_recall":       cor_f1["recall"],
        "corrected_f1":           cor_f1["f1"],

        "corrected_bleu":         cor_bleu,
        "corrected_bleu_1":       cor_bleu_1,

        "sgpt_precision":         s_f1["precision"],
        "sgpt_recall":            s_f1["recall"],
        "sgpt_f1":                s_f1["f1"],
        "sgpt_bleu":              s_bleu,

        "sparql_valid":           true,

        "model":                  llm.model_name
    }

    pd.DataFrame([row]).to_csv(
        csv_file,
        mode='a',
        header=False,
        index=False
    )

    print("Saved to CSV")
    print("=" * 100)

In [21]:
# ========================================================
# SGPT NORMALIZATION
# ========================================================
# ========================================================
# IMPORTS
# ========================================================

import time
import os
import re
import string
import pandas as pd

from collections import Counter

from sacrebleu.metrics import BLEU
from nltk.translate.bleu_score import sentence_bleu


RE_ART  = re.compile(r'\b(a|an|the)\b')
RE_PUNC = re.compile(r'[!"#$%&()*+,-./:;<=>?@\[\]\\^`{|}~_\']')


def _remove_articles(t):
    return RE_ART.sub(' ', t)


def _white_space_fix(t):
    return ' '.join(t.split())


def _remove_punc(t):
    return RE_PUNC.sub(' ', t)


def _lower(t):
    return t.lower()


def normalize(text: str) -> str:
    """
    Lower text and remove punctuation,
    articles and extra whitespace.
    """
    return _white_space_fix(
        _remove_articles(
            _remove_punc(
                _lower(text)
            )
        )
    )


# ========================================================
# BLEU NORMAL
# ========================================================

def bleu_sacre_normal(reference: str, hypothesis: str) -> float:

    bleu = BLEU(effective_order=True)

    result = bleu.sentence_score(
        hypothesis,
        [reference]
    )

    return result.score / 100.0


# ========================================================
# SGPT F1
# ========================================================

def sgpt_f1(reference: str, hypothesis: str) -> dict:

    ref_tokens = normalize(reference).split()
    hyp_tokens = normalize(hypothesis).split()

    if not ref_tokens and not hyp_tokens:
        return {
            "precision": 1.0,
            "recall": 1.0,
            "f1": 1.0
        }

    if not ref_tokens or not hyp_tokens:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1": 0.0
        }

    common_tokens = sum(
        (
            Counter(ref_tokens)
            &
            Counter(hyp_tokens)
        ).values()
    )

    precision = common_tokens / len(hyp_tokens)
    recall    = common_tokens / len(ref_tokens)

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = (
            2 * precision * recall
        ) / (precision + recall)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ========================================================
# CSV SETUP
# ========================================================

csv_file = "combined_lc_quad_2_STG220.csv"

y, x = test_pipeline("output_20.csv")

df_test = pd.read_csv("output_20.csv")

if not os.path.exists(csv_file):

    pd.DataFrame(columns=[

        "question",
        "ground_truth",
        "final_sparql",

        "bleu_normal",

        "sgpt_f1",

        "sparql_valid",

        "model"

    ]).to_csv(csv_file, index=False)


# ========================================================
# AUTO RESUME
# ========================================================

if os.path.exists(csv_file):

    existing  = pd.read_csv(csv_file)
    start_idx = len(existing)

else:

    start_idx = 0


end_idx = len(x)


# ========================================================
# METRIC ACCUMULATORS
# ========================================================

t_bleu_normal = 0

t_sgpt_f1     = 0

validate_sparql = 0


# ========================================================
# TRAIN LOOP
# ========================================================

start = time.time()

for items in range(start_idx, end_idx):

    count = items - start_idx + 1
    print(x[items])
    print(y[items])
    print(df_test["entity"][items],df_test["relations"][items] )

    initial_state = {

        "QUESTION": x[items],

        "KB": "wikidata",

        "URIS": {
            "entity":   df_test["entity"][items],
            "relation": df_test["relations"][items]
        },

        "LLM":  llm,
        "LLM2": llm2
    }

    final_state = CHAIN.invoke(initial_state)

    output = final_state["FINAL_SPARQL"]

    print(f"[GT]           : {y[items]}")
    print(f"[FINAL ANSWER] : {output}")
    print(f"model name {llm.model_name}")

    infer_time = time.time() - start


    # ====================================================
    # BLEU NORMAL
    # ====================================================

    b_normal = bleu_sacre_normal(
        y[items],
        output
    )

    t_bleu_normal += b_normal


    # ====================================================
    # SGPT F1
    # ====================================================

    s_f1 = sgpt_f1(
        y[items],
        output
    )

    t_sgpt_f1 += s_f1["f1"]


    # ====================================================
    # SPARQL VALIDATION
    # ====================================================

    true, _ = check_sparql_query_validity(
        output,
        "wikidata"
    )

    if true:
        validate_sparql += 1


    # ====================================================
    # LOGGING
    # ====================================================

    print(
        f"epoch: {items+1} | "
        f"BLEU_normal: {t_bleu_normal/count:.4f} | "
        f"SGPT_F1: {t_sgpt_f1/count:.4f} | "
        f"valid_sparql: {validate_sparql} | "
        f"time: {infer_time/count:.4f}s"
    )

    print("=" * 100)


    # ====================================================
    # SAVE ROW
    # ====================================================

    row = {

        "question":     x[items],
        "ground_truth": y[items],
        "final_sparql": output,

        "bleu_normal":  b_normal,

        "sgpt_f1":      s_f1["f1"],

        "sparql_valid": true,

        "model":        llm.model_name
    }

    pd.DataFrame([row]).to_csv(
        csv_file,
        mode='a',
        header=False,
        index=False
    )

    print("Saved to CSV")
    print("=" * 100)

NameError: name 'test_pipeline' is not defined